# 5.1 - ViT (ViT-B/16, transfer learning, pré-processamento v2: MTCNN 224, margem 56)

ViT-B/16 pré-treinado, ajustado para real vs. ataque no NUAA seguindo George & Marcel, com as imagens do
pré-processamento v2 (224x224 RGB, sem redimensionar). **Um notebook só roda o nível de ajuste do artigo
(só a camada final treina), com 3 seeds (3 treinos).** Nada do v1 nem do CNN é tocado: tudo é gravado em `TCC/v2/`.

| Nível (`exp_id`) | O que treina |
|---|---|
| `vit_b16_v2_01_fc` | só a camada final (FC 768 -> 1); o resto do ViT fica congelado |

Cada treino é avaliado na validação; o teste só roda na seção final, uma vez por seed.

## Decisões e origem

Base: George & Marcel, "On the Effectiveness of Vision Transformers for Zero-shot Face Anti-Spoofing" (arXiv 2011.08019).

- **Modelo:** ViT "base", patch 16x16, 224x224 (George & Marcel)
- **Biblioteca:** timm, versão fixada no notebook (George & Marcel usam o timm; a versão é decisão do projeto)
- **Pesos:** `orig_in21k_ft_in1k` (decisão do projeto; o artigo não informa o checkpoint, só diz "grandes quantidades de dados").
  São os pesos originais do ViT (código oficial do Google), pré-treinados no ImageNet-21k e ajustados no ImageNet-1k, segundo os comentários do timm 1.0.30
- **Entrada:** 224x224 RGB, sem resize (George & Marcel usam 224x224 com MTCNN)
- **Normalização:** a do checkpoint, lida da configuração dos pesos (decisão do projeto; nesses pesos é média 0,5 e desvio 0,5, diferente da do ImageNet clássico usada no CNN)
- **Cabeça:** FC 768 -> 1 logit, sem camada oculta e sem dropout, sigmoide = P(spoof) (George & Marcel)
- **Nível:** só a FC treina (George & Marcel)
- **Otimizador e learning rate:** Adam, lr 1e-4 fixo, weight decay 1e-5 (George & Marcel)
- **Perda:** BCE (George & Marcel)
- **Épocas:** 20 fixas; o modelo escolhido é o de menor loss de validação; sem early stopping (George & Marcel)
- **Batch:** 16 no treino (George & Marcel); 32 na avaliação, igual ao CNN, para o tempo de predição ser comparável (decisão do projeto)
- **Augmentation:** só flip horizontal com p = 0,5 (George & Marcel); rotação e brilho do CNN ficam desligados
- **Precisão mista (AMP):** ligada, como no CNN (decisão do projeto; o artigo não informa)
- **Seeds:** 42, 43 e 44, média ± desvio (decisão do projeto; repetir o treino: Bouthillier et al., 2021; Dodge et al., 2020)
- **Threshold, métricas, teste, tempo por seed:** iguais ao CNN (decisão do projeto)

## Limitações a registrar

- George & Marcel não informam o checkpoint exato do timm; o usado aqui é escolha do projeto.
- O artigo não usa o NUAA: avalia ataques não vistos (HQ-WMCA, SiW-M) e entre bancos. O resultado dele não prevê o resultado no NUAA.
- Parâmetros diferentes do CNN (cabeça, parada, augmentation, batch). O protocolo (dados, split, métricas, teste) é o mesmo.
- No artigo, as CNNs comparadas (ResNet101, DenseNet161) só treinam a última camada e os parâmetros delas não são informados.
- Só o nível de FC está incluído. No CNN, só a cabeça foi muito pior que liberar o último bloco; um nível parcial do ViT seria decisão do projeto, sem base direta em artigo.
- A validação do v1 e do CNN ficou saturada (HTER perto de 0%), então ela pode não separar configurações.
- As seeds não tornam o treino 100% reprodutível (`set_seed` não fixa o cuDNN e o AMP está ligado). Com o backbone congelado e sem dropout, só a cabeça e a ordem dos dados variam entre seeds.
- Cerca de 30% dos recortes têm menos de 224 px na origem e foram ampliados pelo MTCNN.
- 3 seeds é uma restrição de custo computacional.
- O tempo depende da GPU que o Colab entregar; o hardware fica registrado em cada linha.

## Saídas (pastas novas em `TCC/v2/`)

- imagens em cache: `v2/cache/cnn_224/` (o mesmo cache do CNN; as imagens são as mesmas)
- checkpoints: `v2/models/vit_b16/<exp_id>.pt` (só os parâmetros que treinam)
- por treino: `v2/experiments/vit_b16/runs/<exp_id>/` (`history.csv`, `run_info.json`, `predictions_test.csv`)
- log de resultados: `v2/experiment_log_v2.csv` (o mesmo do LBP, HOG e CNN; só acrescenta linhas)
- tempos: `v2/timing_v2.csv` (uma linha de validação e uma de teste por seed)

**Pré-requisito:** copiar `vit_v2.py` para `src/modeling/` no repositório e subir para o GitHub
(`src/timing.py` e `src/modeling/cnn_v2.py` já estão lá).

**Se o Colab desconectar:** rode as células de configuração e a de treino de novo. Os treinos já concluídos são lidos de
`run_info.json` e não são refeitos.

In [ ]:
# Clona o repositório (ou reaproveita, se já existir) e instala as dependências.
# facenet-pytorch precisa de --no-deps (ver 2.0). Este notebook não roda MTCNN,
# mas src/dataset.py importa o facenet-pytorch.
import os
if not os.path.isdir('/content/liveness-detection'):
    !git clone https://github.com/laianemuckler/liveness-detection.git
%cd /content/liveness-detection
!git pull --quiet

!pip install -r requirements.txt --quiet
!pip install facenet-pytorch==2.6.0 --no-deps --quiet
!pip install timm==1.0.30 --quiet

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import csv
import json
from datetime import date

import numpy as np
import pandas as pd
import timm
import torch
from IPython.display import display

from src.config import (
    TRAIN_V2_DIR, VAL_V2_DIR, TEST_V2_DIR,
    DRIVE_ROOT, LABEL_LIVE, LABEL_SPOOF,
)
from src.dataset import load_processed_split
from src.modeling.predict import compute_metrics
from src.modeling.cnn import (
    set_seed, FASDataset, make_loader, count_parameters, load_images,
)
from src.modeling.vit_v2 import (
    LEVELS, build_vit_v2, get_norm, get_transforms_vit, train_vit_v2,
    load_trainable_state, resource_check, evaluate_cnn_timed, preprocess_one,
)
from src.timing import (
    medir_por_item, copiar_amostra, gravar_tempo, gravar_tempo_teste, hardware_info,
)

# ---------------- Hiperparâmetros (ver as decisões no topo do notebook) ----------------
LEVELS_TO_RUN = list(LEVELS)
SEEDS = [42, 43, 44]
MODEL_NAME = 'vit_base_patch16_224'
WEIGHTS_TAG = 'orig_in21k_ft_in1k' # checkpoint do timm (decisão do projeto)
BATCH_SIZE = 16                    # treino (George & Marcel)
EVAL_BATCH_SIZE = 32               # validação e teste (igual ao CNN)
MAX_EPOCHS = 20                    # épocas fixas (George & Marcel)
PATIENCE = None                    # None = sem early stopping; o melhor modelo é o de menor loss de validação
LR_HEAD = 1e-4                     # George & Marcel
WEIGHT_DECAY = 1e-5                # George & Marcel
FLIP_P = 0.5                       # George & Marcel
ROTATION_DEGREES = 0               # desligado (o CNN usa 10)
BRIGHTNESS = 0.0                   # desligado (o CNN usa 0,2)
THRESHOLD = 0.5                    # limiar fixo na saída sigmoide

TIMM_VERSION = timm.__version__

# Contagens do protocolo (live, spoof)
EXPECTED_COUNTS = {'train': (1407, 1366), 'val': (336, 382), 'test': (3362, 5761)}

# ---------------- Saídas do v2 (pastas novas, nada do v1 nem do CNN é tocado) ----------------
OUTPUT_V2_ROOT = os.path.join(DRIVE_ROOT, 'v2')
RUNS_DIR = os.path.join(OUTPUT_V2_ROOT, 'experiments', 'vit_b16', 'runs')
CKPT_DIR = os.path.join(OUTPUT_V2_ROOT, 'models', 'vit_b16')
CACHE_DIR = os.path.join(OUTPUT_V2_ROOT, 'cache', 'cnn_224')   # mesmo cache do CNN (mesmas imagens)
LOG_V2_PATH = os.path.join(OUTPUT_V2_ROOT, 'experiment_log_v2.csv')
TIMING_V2_PATH = os.path.join(OUTPUT_V2_ROOT, 'timing_v2.csv')
for _d in (RUNS_DIR, CKPT_DIR, CACHE_DIR):
    os.makedirs(_d, exist_ok=True)

# Textos gravados no log (iguais nas linhas de validação e de teste)
METODO = 'ViT-B/16 (v2 224)'
FEATURE_CONFIG = f'RGB 224x224, {MODEL_NAME}.{WEIGHTS_TAG} (timm {TIMM_VERSION})'


def modelo_config(level):
    aug = f"flip{FLIP_P}" + (f" rot{ROTATION_DEGREES}" if ROTATION_DEGREES else '') + \
          (f" brilho{BRIGHTNESS}" if BRIGHTNESS else '')
    parada = f"paciencia={PATIENCE}" if PATIENCE is not None else "sem early stopping (melhor loss val)"
    return (f"adam lr_head={LR_HEAD} wd={WEIGHT_DECAY}, so a FC treina, cabeca=768->1, "
            f"batch_treino={BATCH_SIZE} batch_aval={EVAL_BATCH_SIZE}, epocas={MAX_EPOCHS}, {parada}, aug={aug}")


device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device} | timm {TIMM_VERSION}")
if device.type != 'cuda':
    print("AVISO: sem GPU. Ative a GPU no Colab (Ambiente de execução > Alterar o tipo) antes de treinar.")

In [ ]:
# Log do v2: mesmas colunas e formatação do log do LBP, HOG e CNN v2,
# gravando em TCC/v2/experiment_log_v2.csv (o log do v1 não é tocado).

def log_experiment_v2(exp_id, metodo, feature_config, modelo_config,
                      hter_val=None, auc_val=None, hter_test=None, auc_test=None,
                      eer_val=None, eer_test=None, obs=''):
    def fmt(v):
        return f"{v:.4f}" if v is not None else ''

    row = {
        'exp_id': exp_id, 'metodo': metodo, 'feature_config': feature_config,
        'modelo_config': modelo_config,
        'HTER_val': fmt(hter_val), 'AUC_val': fmt(auc_val), 'EER_val': fmt(eer_val),
        'HTER_test': fmt(hter_test), 'AUC_test': fmt(auc_test), 'EER_test': fmt(eer_test),
        'data': date.today().isoformat(), 'obs': obs,
    }
    file_exists = os.path.isfile(LOG_V2_PATH)
    os.makedirs(os.path.dirname(LOG_V2_PATH), exist_ok=True)
    with open(LOG_V2_PATH, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=list(row.keys()))
        if not file_exists:
            writer.writeheader()
        writer.writerow(row)
    print(f"Registrado '{exp_id}' em {LOG_V2_PATH}")


def ja_registrado(exp_id):
    """True se esse exp_id já está no log do v2 (evita duplicar linhas ao rodar de novo)."""
    if not os.path.isfile(LOG_V2_PATH):
        return False
    return exp_id in set(pd.read_csv(LOG_V2_PATH, dtype=str)['exp_id'])

In [ ]:
# Carrega as listas de imagens v2 (treino e validação) e confere com o protocolo

train_paths, y_train = load_processed_split(TRAIN_V2_DIR)
val_paths, y_val = load_processed_split(VAL_V2_DIR)

print(f"Train: {len(train_paths)} | Validation: {len(val_paths)}")

for nome, y_, esperado in (('train', y_train, EXPECTED_COUNTS['train']),
                           ('val', y_val, EXPECTED_COUNTS['val'])):
    y_ = np.array(y_)
    encontrado = (int(np.sum(y_ == LABEL_LIVE)), int(np.sum(y_ == LABEL_SPOOF)))
    print(f"  {nome:<5} live/spoof = {encontrado}")
    if encontrado != esperado:
        raise ValueError(f"Contagem de '{nome}' não bate com o protocolo: "
                         f"esperado {esperado}, encontrado {encontrado}.")

# garante que são as imagens v2 (224) e não as do v1 (160)
from PIL import Image
with Image.open(train_paths[0]) as im0:
    if im0.size != (224, 224):
        raise ValueError(f"Imagem com tamanho {im0.size}; o esperado no v2 é (224, 224).")
print("Imagens v2 conferidas (224x224).")

In [ ]:
# Normalização do checkpoint (lida da configuração dos pesos), imagens na memória e dataloaders.
# Na primeira vez lê do Drive e salva um arquivo de cache (.npz); se o CNN já foi rodado, o cache existe.
import time

t0 = time.time()
_ref = build_vit_v2(LEVELS_TO_RUN[0], MODEL_NAME, WEIGHTS_TAG, pretrained=False)
NORM_MEAN, NORM_STD, INPUT_SIZE = get_norm(_ref)
del _ref
print(f"Normalização do checkpoint: média {NORM_MEAN}, desvio {NORM_STD}, entrada {INPUT_SIZE}")
if INPUT_SIZE != (3, 224, 224):
    raise ValueError(f"O checkpoint espera entrada {INPUT_SIZE}; os recortes v2 são 224x224.")

train_tf, eval_tf = get_transforms_vit(NORM_MEAN, NORM_STD, FLIP_P, ROTATION_DEGREES, BRIGHTNESS)

train_images = load_images(train_paths, os.path.join(CACHE_DIR, 'train.npz'))
val_images = load_images(val_paths, os.path.join(CACHE_DIR, 'val.npz'))

train_ds = FASDataset(train_images, y_train, train_tf)
val_ds = FASDataset(val_images, y_val, eval_tf)

val_loader = make_loader(val_ds, EVAL_BATCH_SIZE, shuffle=False)
print(f"Dados prontos em {time.time() - t0:.1f}s")

In [ ]:
# Confere quantos parâmetros treinam em cada nível (sem baixar os pesos)

for level in LEVELS_TO_RUN:
    m = build_vit_v2(level, MODEL_NAME, WEIGHTS_TAG, pretrained=False)
    total, trainable = count_parameters(m)
    print(f"{level}: {trainable:,} de {total:,} parâmetros treináveis ({100*trainable/total:.4f}%)")
    del m

In [ ]:
# Tempo de "extração" do ViT: ler a imagem do disco + pré-processamento de avaliação, por imagem.
# Medido uma vez, em 200 imagens da validação copiadas para o disco local (não inclui a leitura
# lenta do Drive), com 1 processo. O mesmo valor entra na linha de tempo de cada seed.
# A rede em si (predição) é medida em evaluate_cnn_timed, na GPU.

N_AMOSTRA_TEMPO = 200
amostra = copiar_amostra(val_paths, N_AMOSTRA_TEMPO, '/content/amostra_tempo_vit')
EXTRACAO_MS = medir_por_item(lambda p: preprocess_one(p, eval_tf), amostra)
print(f"Extração (ler + pré-processar): {EXTRACAO_MS:.2f} ms/img "
      f"({len(amostra)} imagens da validação, 1 processo)")

## Verificação de memória e tempo (opcional, antes de treinar)

Roda poucos passos de treino com o batch do treino, só para ver o pico de memória da GPU e o tempo por passo.
Não treina de verdade nem grava nada. O tempo por época que aparece é uma **estimativa** (sem a validação).
Se der falta de memória, reduza `BATCH_SIZE` na configuração e rode de novo (e registre a mudança, porque o
George & Marcel usam 16).

In [ ]:
RUN_RESOURCE_CHECK = True

if RUN_RESOURCE_CHECK:
    if device.type != 'cuda':
        raise RuntimeError("Sem GPU: ative a GPU no Colab antes.")
    set_seed(SEEDS[0])
    _m = build_vit_v2(LEVELS_TO_RUN[0], MODEL_NAME, WEIGHTS_TAG)
    _loader = make_loader(train_ds, BATCH_SIZE, shuffle=True, seed=0)
    chk = resource_check(_m, _loader, device, lr_head=LR_HEAD, weight_decay=WEIGHT_DECAY)
    if chk['oom']:
        print(f"SEM MEMÓRIA com batch {BATCH_SIZE} (pico {chk['peak_mem_gb']:.2f} GB). Reduza BATCH_SIZE.")
    else:
        print(f"Pico de memória: {chk['peak_mem_gb']:.2f} GB de {chk['gpu_total_gb']:.1f} GB")
        print(f"Tempo por passo: {chk['s_per_step']:.3f} s | estimativa de uma época de treino: "
              f"{chk['est_epoch_s']:.0f} s ({chk['est_epoch_s']/60:.1f} min), sem a validação")
        print(f"Estimativa de {MAX_EPOCHS} épocas por seed: {MAX_EPOCHS*chk['est_epoch_s']/60:.0f} min "
              f"(só treino; a validação a cada época soma mais)")
    del _m, _loader
    torch.cuda.empty_cache()

## Treino e validação

Um treino por seed (3 no total). Em cada seed, o checkpoint de menor loss de validação é salvo no Drive.
A validação serve só para escolher o melhor modelo de cada treino e para acompanhar; o teste fica para a última seção.

Para cada treino ficam registrados: tempo total de treino, épocas executadas, melhor época e tempo de
predição por imagem (só a rede, na GPU). O tempo de treino inclui a validação feita a cada época.

In [ ]:
# Treina cada nível com cada seed, avalia na validação e registra no log e no tempo.
# Retomável: se o treino de um (nível, seed) já terminou, ele é lido de run_info.json e não é refeito.

if device.type != 'cuda':
    raise RuntimeError("Sem GPU: não treine na CPU (demora demais e o tempo medido não serve). "
                       "Ative a GPU no Colab e rode as células de configuração de novo.")

HARDWARE = hardware_info(usado='GPU (treino e predição da rede); CPU (leitura e pré-processamento)')
print(HARDWARE)

resultados_val = []

for level in LEVELS_TO_RUN:
    for seed in SEEDS:
        exp_id = f"{level}_seed{seed}"
        exp_dir = os.path.join(RUNS_DIR, exp_id)
        ckpt_path = os.path.join(CKPT_DIR, f"{exp_id}.pt")
        info_path = os.path.join(exp_dir, 'run_info.json')
        os.makedirs(exp_dir, exist_ok=True)

        if os.path.exists(info_path):
            with open(info_path) as f:
                info = json.load(f)
            print(f"[{exp_id}] já treinado ({info['epochs']} épocas, {info['train_s']/60:.1f} min); "
                  f"lido de run_info.json")
        else:
            if os.path.exists(ckpt_path):
                raise RuntimeError(f"Existe {ckpt_path} mas não {info_path} (treino interrompido depois de "
                                   f"salvar o checkpoint). Não vou sobrescrever: confira a pasta antes de continuar.")
            print(f"\n===== {exp_id} =====")
            set_seed(seed)
            model = build_vit_v2(level, MODEL_NAME, WEIGHTS_TAG)
            train_loader = make_loader(train_ds, BATCH_SIZE, shuffle=True, seed=seed)

            model, history, train_info = train_vit_v2(
                model, train_loader, val_loader, y_val, device,
                lr_head=LR_HEAD, weight_decay=WEIGHT_DECAY,
                max_epochs=MAX_EPOCHS, patience=PATIENCE,
                checkpoint_path=ckpt_path,
            )
            history.to_csv(os.path.join(exp_dir, 'history.csv'), index=False)

            val_res = evaluate_cnn_timed(model, val_loader, y_val, device, threshold=THRESHOLD)
            print(f"Validação -> HTER: {val_res['HTER']*100:.2f}% | AUC: {val_res['AUC']:.4f} | "
                  f"EER: {val_res['EER']*100:.2f}% | FAR: {val_res['FAR']*100:.2f}% | FRR: {val_res['FRR']*100:.2f}%")

            info = {
                **train_info,
                'HTER': float(val_res['HTER']), 'AUC': float(val_res['AUC']), 'EER': float(val_res['EER']),
                'FAR': float(val_res['FAR']), 'FRR': float(val_res['FRR']), 'val_loss': float(val_res['loss']),
                'n_val': int(val_res['n']),
                'predicao_ms_por_img_val': float(val_res['forward_s'] / val_res['n'] * 1000),
                'hardware': HARDWARE,
            }
            with open(info_path, 'w') as f:
                json.dump(info, f, indent=2)
            del model
            torch.cuda.empty_cache()

        # Registros (não duplicam se já existirem)
        if ja_registrado(exp_id):
            print(f"[{exp_id}] já está no log do v2; não registrei de novo.")
        else:
            log_experiment_v2(
                exp_id=exp_id, metodo=METODO, feature_config=FEATURE_CONFIG,
                modelo_config=f"{modelo_config(level)}, seed={seed}",
                hter_val=info['HTER'], auc_val=info['AUC'], eer_val=info['EER'],
                obs=f"pre-processamento v2 224; {info['epochs']} epocas (melhor: {info['best_epoch']})",
            )
        gravar_tempo(
            TIMING_V2_PATH, exp_id, 'validacao', info['n_val'],
            extracao_ms=EXTRACAO_MS, treino_s=info['train_s'], predicao_ms=info['predicao_ms_por_img_val'],
            hardware=info['hardware'],
            obs=(f"epocas={info['epochs']}, melhor_epoca={info['best_epoch']}, "
                 f"{info['train_s']/info['epochs']:.1f}s/epoca; treino inclui a validacao a cada epoca; "
                 f"predicao = so a rede, lote de {EVAL_BATCH_SIZE}, AMP; "
                 f"extracao = ler + pre-processar ({N_AMOSTRA_TEMPO} imagens da validacao, 1 processo)"),
        )

        resultados_val.append({
            'nivel': level, 'seed': seed, 'HTER': info['HTER'], 'AUC': info['AUC'], 'EER': info['EER'],
            'epocas': info['epochs'], 'melhor_epoca': info['best_epoch'],
            'treino_min': info['train_s'] / 60, 'pred_ms_img': info['predicao_ms_por_img_val'],
        })

In [ ]:
# Resumo da validação por nível (média e desvio entre seeds) e linha do nível no log

df_val = pd.DataFrame(resultados_val)
display(df_val)

for level, g in df_val.groupby('nivel', sort=False):
    print(f"\n{level}  ({len(g)} seeds)")
    print(f"  HTER val: {g.HTER.mean()*100:.2f}% +- {g.HTER.std()*100:.2f}")
    print(f"  AUC val:  {g.AUC.mean():.4f} +- {g.AUC.std():.4f}")
    print(f"  EER val:  {g.EER.mean()*100:.2f}% +- {g.EER.std()*100:.2f}")
    print(f"  Treino por seed: {g.treino_min.round(1).tolist()} min | épocas: {g.epocas.tolist()} "
          f"| melhor época: {g.melhor_epoca.tolist()}")
    if ja_registrado(level):
        print(f"  [{level}] já está no log do v2; não registrei de novo.")
    else:
        log_experiment_v2(
            exp_id=level, metodo=METODO, feature_config=FEATURE_CONFIG, modelo_config=modelo_config(level),
            hter_val=g.HTER.mean(), auc_val=g.AUC.mean(), eer_val=g.EER.mean(),
            obs=f"Media de {len(g)} seeds (desvio HTER {g.HTER.std()*100:.2f} p.p.)",
        )

## Teste (rodar somente no final)

Só rode esta seção depois de olhar a validação. Ela usa os checkpoints salvos e avalia
**cada seed uma única vez** no conjunto de teste, sem escolher uma configuração pelo resultado do teste.
Mantenha `RUN_TEST = False` até lá.

Se a seção for interrompida, ao rodar de novo as seeds já avaliadas não são avaliadas outra vez: as métricas são
recalculadas das predições salvas em `predictions_test.csv`.

In [ ]:
RUN_TEST = False

In [ ]:
if RUN_TEST:
    # Define de novo o que o teste precisa, para funcionar depois de reiniciar o runtime
    # (as células de configuração e de funções devem ter sido rodadas).
    _ref = build_vit_v2(LEVELS_TO_RUN[0], MODEL_NAME, WEIGHTS_TAG, pretrained=False)
    NORM_MEAN, NORM_STD, INPUT_SIZE = get_norm(_ref)
    del _ref
    _, eval_tf = get_transforms_vit(NORM_MEAN, NORM_STD, FLIP_P, ROTATION_DEGREES, BRIGHTNESS)
    HARDWARE = hardware_info(usado='GPU (treino e predição da rede); CPU (leitura e pré-processamento)')

    test_paths, y_test = load_processed_split(TEST_V2_DIR)
    y_t = np.array(y_test)
    encontrado = (int(np.sum(y_t == LABEL_LIVE)), int(np.sum(y_t == LABEL_SPOOF)))
    if encontrado != EXPECTED_COUNTS['test']:
        raise ValueError(f"Contagem do teste não bate com o protocolo: esperado {EXPECTED_COUNTS['test']}, "
                         f"encontrado {encontrado}.")
    test_images = load_images(test_paths, os.path.join(CACHE_DIR, 'test.npz'))
    test_ds = FASDataset(test_images, y_test, eval_tf)
    test_loader = make_loader(test_ds, EVAL_BATCH_SIZE, shuffle=False)
    print(f"Test: {len(test_paths)}")

    resultados_teste = []
    for level in LEVELS_TO_RUN:
        for seed in SEEDS:
            exp_id = f"{level}_seed{seed}"
            final_id = exp_id + '_FINAL_TEST'
            exp_dir = os.path.join(RUNS_DIR, exp_id)
            pred_path = os.path.join(exp_dir, 'predictions_test.csv')
            forward_s = None

            if os.path.exists(pred_path):
                dfp = pd.read_csv(pred_path)
                if len(dfp) != len(test_paths):
                    raise RuntimeError(f"{pred_path} tem {len(dfp)} linhas; esperado {len(test_paths)}. "
                                       f"Não vou sobrescrever: confira a pasta.")
                scores = dfp['score'].values
                res = compute_metrics(y_t, scores, (scores >= THRESHOLD).astype(int))
                print(f"[{exp_id}] já avaliado no teste; métricas recalculadas das predições salvas "
                      f"(o teste não foi rodado de novo).")
            else:
                model = build_vit_v2(level, MODEL_NAME, WEIGHTS_TAG)   # pesos pré-treinados do timm
                load_trainable_state(model, os.path.join(CKPT_DIR, f"{exp_id}.pt"))
                model.to(device)
                res = evaluate_cnn_timed(model, test_loader, y_test, device, threshold=THRESHOLD,
                                         desc=f"teste {exp_id}")
                forward_s = res['forward_s']
                pd.DataFrame({
                    'image_path': test_paths,
                    'label_true': y_test,
                    'label_pred': res['y_pred'],
                    'score': res['y_scores'],
                }).to_csv(pred_path, index=False)
                del model
                torch.cuda.empty_cache()

            print(f"{exp_id} -> HTER: {res['HTER']*100:.2f}% | AUC: {res['AUC']:.4f} | "
                  f"EER: {res['EER']*100:.2f}% | FAR: {res['FAR']*100:.2f}% | FRR: {res['FRR']*100:.2f}%")

            if not ja_registrado(final_id):
                log_experiment_v2(
                    exp_id=final_id, metodo=METODO, feature_config=FEATURE_CONFIG,
                    modelo_config=f"{modelo_config(level)}, seed={seed}",
                    hter_test=res['HTER'], auc_test=res['AUC'], eer_test=res['EER'],
                    obs='Métrica de teste de uma seed, avaliada uma única vez',
                )
            if forward_s is not None:
                gravar_tempo_teste(TIMING_V2_PATH, exp_id, len(test_paths), forward_s, hardware=HARDWARE)

            resultados_teste.append({'nivel': level, 'seed': seed, 'HTER': res['HTER'], 'AUC': res['AUC'],
                                     'EER': res['EER'], 'FAR': res['FAR'], 'FRR': res['FRR']})

    df_test = pd.DataFrame(resultados_teste)
    display(df_test)
    for level, g in df_test.groupby('nivel', sort=False):
        print(f"\n{level}  ({len(g)} seeds)")
        print(f"  HTER teste: {g.HTER.mean()*100:.2f}% +- {g.HTER.std()*100:.2f}")
        print(f"  AUC teste:  {g.AUC.mean():.4f} +- {g.AUC.std():.4f}")
        print(f"  EER teste:  {g.EER.mean()*100:.2f}% +- {g.EER.std()*100:.2f}  "
              f"(usa o melhor limiar do próprio teste, então é otimista)")
        print(f"  FAR teste:  {g.FAR.mean()*100:.2f}% | FRR teste: {g.FRR.mean()*100:.2f}%")
        level_final = level + '_FINAL_TEST'
        if ja_registrado(level_final):
            print(f"  [{level_final}] já está no log do v2; não registrei de novo.")
        else:
            log_experiment_v2(
                exp_id=level_final, metodo=METODO, feature_config=FEATURE_CONFIG, modelo_config=modelo_config(level),
                hter_test=g.HTER.mean(), auc_test=g.AUC.mean(), eer_test=g.EER.mean(),
                obs=f"Métrica final de teste do nível: média de {len(g)} seeds (desvio HTER {g.HTER.std()*100:.2f} p.p.)",
            )
else:
    print("RUN_TEST = False: teste não executado.")